In [1]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from translation_dataset import create_dataloader, TranslationDataset
from prefix_t5_model import PrefixT5
from trainer import Trainer, TrainingHyperParams, load_model
from prediction import generate_translation
from configs import CHECKPOINT, DATASET, BATCH_SIZE

/home/bina/Documents/qazaq_llm/qazaq_llm/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
t5_tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
t5_model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


## Data Preprocessing

In [3]:
# Load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Split the dataset into train, valid, test (train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])

In [4]:
train_translation_dataset = TranslationDataset(
    data=train_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

valid_translation_dataset = TranslationDataset(
    data=valid_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

test_translation_dataset = TranslationDataset(
    data=test_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)


In [5]:
train_loader = create_dataloader(
    dataset=train_translation_dataset, 
    batch_size=BATCH_SIZE
)

valid_loader = create_dataloader(
    dataset=valid_translation_dataset, 
    batch_size=BATCH_SIZE
)

test_loader = create_dataloader(
    dataset=test_translation_dataset, 
    batch_size=BATCH_SIZE, 
)

In [6]:
print(f"Number of batches in train: {len(train_loader)}")
print(f"Number of batches in valid: {len(valid_loader)}")
print(f"Number of batches in test: {len(test_loader)}")

Number of batches in train: 6434
Number of batches in valid: 715
Number of batches in test: 795


In [7]:
# Get a sample batch
sample_batch = next(iter(train_loader))
    
print("\nSample batch shapes:")
for key, value in sample_batch.items():
    print(f"{key}: {value.shape}")

# Decode a sample from the batch
print("\n")
print("Input:", t5_tokenizer.decode(sample_batch['input_ids'][0].tolist(), skip_special_tokens=True))
print("Target:", t5_tokenizer.decode(sample_batch['labels'][0].tolist(), skip_special_tokens=True))


Sample batch shapes:
input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16, 128])
decoder_input_ids: torch.Size([16, 128])


Input: translate English to French: – Le second cocher est mon ami.
Target: "The second coachman is a friend of mine."


## Prefix T5 Model

In [8]:
prefixed_t5_model = PrefixT5(
    t5_model=t5_model, prefix_length=10, prefix_projection_size=512
)

## Training

In [9]:
sample_data = dataset.select(range(50))
sample_valid = dataset.select(range(50, 70))
sample_trans_data = TranslationDataset(
    data=sample_data,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_loader = create_dataloader(sample_trans_data)

sample_valid_trans_data = TranslationDataset(
    data=sample_valid,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_valid_loader = create_dataloader(sample_valid_trans_data)

In [10]:
training_params = TrainingHyperParams(
    num_epochs=1, 
    batch_size=16, 
    learning_rate=0.001, 
    warmup_steps=100, 
    max_grad_norm=1,
)

trainer = Trainer(
    model=prefixed_t5_model,
    tokenizer=t5_tokenizer,
    train_loader=train_loader,
    valid_loader=valid_loader,
    test_loader=test_loader,
    args=training_params
)

In [11]:
trainer.model

PrefixT5(
  (t5): T5ForConditionalGeneration(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_features=2048,

In [12]:
train_loss, val_loss = trainer.train()

Validation: 100%|██████████| 715/715 [01:06<00:00, 10.68batch/s]


Epoch 1/1
	Train Loss: 0.7210
	Validation Loss: 0.5979

Model saved at models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cuda.pth


In [22]:
!mkdir -p models

In [4]:
# saved_model = load_model(base_model=t5_model, save_path="models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cpu.pth")
# saved_model 
saved_model = load_model(base_model=t5_model, save_path="models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cuda.pth")
saved_model 

Model successfully loaded from models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cuda.pth


/home/bina/Documents/qazaq_llm/qazaq_llm/.conda/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


PrefixT5(
  (t5): T5ForConditionalGeneration(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_features=2048,

In [5]:
res = generate_translation(
    model=saved_model,
    tokenizer=t5_tokenizer, 
    input_text="Hello, my name is Zeinab",
    max_length=2,
)
res

''